<a href="https://colab.research.google.com/github/neeeeeeelpatel/Realtime-Queue-monitoring/blob/main/Automated_Queue_monitoring_system.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
pip install ultralytics opencv-python numpy


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 24.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 74.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 57.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 41.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 41.2 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstalling n

# step - 1 -- apllying yolo and setting the frames to 300 second or 5 minutes and also detecting only humans ( output - 0)
Applying Yolo coz of object detection

applying specifically yolov8s.pt coz its faster and kind of accuarte

this was best yolov8m.pt but cant apply as it needs RTX graphics card

In [ ]:
from ultralytics import YOLO
import cv2


Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [ ]:
model = YOLO("yolov8s.pt")

100%|██████████| 21.5M/21.5M [00:00<00:00, 172MB/s]


In [ ]:
from ultralytics import YOLO
import cv2
import os
import numpy as np

model = YOLO("yolov8s.pt")

# ✅ Use full absolute path for your video
video_path = "queue.mp4"
cap = cv2.VideoCapture(video_path)

# ✅ Confirm the video loaded
if not cap.isOpened():
    print("❌ Video not loaded. Check path or format:", video_path)
    exit()

fps = cap.get(cv2.CAP_PROP_FPS)
if fps == 0:
    print("❌ FPS is 0. Video failed to load.")
    exit()

print("🎞️ FPS:", fps)

# 1 frame per second
interval_seconds = 1
frames_to_skip = int(fps * interval_seconds)

# ✅ Save path
saved_frames = "saved_frames"
os.makedirs(saved_frames, exist_ok=True)

# ✅ Test dummy save
dummy = np.zeros((100, 100, 3), dtype=np.uint8)
test_path = os.path.join(saved_frames, "test_save.png")
cv2.imwrite(test_path, dummy)

frame_id = 0

while cap.isOpened():
    success, frame = cap.read()
    if not success:
        break

    if frame_id % frames_to_skip == 0:
        minutes = int(frame_id / fps / 60)
        seconds = int(frame_id / fps % 60)
        print(f"\n⏱️ Frame [{frame_id}] at {minutes:02d}:{seconds:02d}")

        results = model(frame)
        person_count = 0
        for box in results[0].boxes:
            if int(box.cls[0]) == 0:
                person_count += 1
                x1, y1, x2, y2 = map(int, box.xyxy[0])
                cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)

        print(f"🧍 People detected: {person_count}")

        filename = os.path.join(saved_frames, f"frame_{minutes:02d}_{seconds:02d}.png")
        if cv2.imwrite(filename, frame):
            print(f" Saved frame: {filename}")
        else:
            print(" Failed to save frame!")

    frame_id += 1

cap.release()
print(" Video processing complete.")


🎞️ FPS: 25.0

⏱️ Frame [0] at 00:00

0: 384x640 10 persons, 2 handbags, 1 suitcase, 406.0ms
Speed: 5.6ms preprocess, 406.0ms inference, 1.6ms postprocess per image at shape (1, 3, 384, 640)
🧍 People detected: 10
💾 Saved frame: saved_frames/frame_00_00.png

⏱️ Frame [25] at 00:01

0: 384x640 14 persons, 1 backpack, 1 handbag, 384.0ms
Speed: 4.2ms preprocess, 384.0ms inference, 2.1ms postprocess per image at shape (1, 3, 384, 640)
🧍 People detected: 14
💾 Saved frame: saved_frames/frame_00_01.png

⏱️ Frame [50] at 00:02

0: 384x640 13 persons, 1 dog, 1 handbag, 379.2ms
Speed: 4.1ms preprocess, 379.2ms inference, 1.4ms postprocess per image at shape (1, 3, 384, 640)
🧍 People detected: 13
💾 Saved frame: saved_frames/frame_00_02.png

⏱️ Frame [75] at 00:03

0: 384x640 13 persons, 1 handbag, 574.7ms
Speed: 4.1ms preprocess, 574.7ms inference, 1.9ms postprocess per image at shape (1, 3, 384, 640)
🧍 People detected: 13
💾 Saved frame: saved_frames/frame_00_03.png

⏱️ Frame [100] at 00:04

0: 384

In [ ]:
from google.colab import files

to save it in our computer there is a slit code do it if u need

os for saving files and pictures related to files

## step - 2
for low light
You don’t need heavy enhancements like night vision or IR filters — just mild contrast/brightness improvements to handle real-world variations.

so we will use **CLAHE-based contrast enhancement,**
this is the method we are using
**Contrast Limited Adaptive Histogram Equalization**



In [ ]:
from ultralytics import YOLO
import cv2
import os
import numpy as np
#used for low light
def enhance_contrast_brightness(frame):
    lab = cv2.cvtColor(frame, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)

    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    l_clahe = clahe.apply(l)

    merged = cv2.merge((l_clahe, a, b))
    enhanced_frame = cv2.cvtColor(merged, cv2.COLOR_LAB2BGR)
    return enhanced_frame
#low light ends here
model = YOLO("yolov8s.pt")

# ✅ Use full absolute path for your video
video_path = "queue.mp4"
cap = cv2.VideoCapture(video_path)

# ✅ Confirm the video loaded
if not cap.isOpened():
    print(" Video not loaded. Check path or format:", video_path)
    exit()

fps = cap.get(cv2.CAP_PROP_FPS)
if fps == 0:
    print("❌ FPS is 0. Video failed to load.")
    exit()

print("🎞️ FPS:", fps)

# 1 frame per second
interval_seconds = 1
frames_to_skip = int(fps * interval_seconds)

# ✅ Save path
saved_frames = "saved_frames"
os.makedirs(saved_frames, exist_ok=True)

# ✅ Test dummy save
dummy = np.zeros((100, 100, 3), dtype=np.uint8)
test_path = os.path.join(saved_frames, "test_save.png")
cv2.imwrite(test_path, dummy)

frame_id = 0

while cap.isOpened():
    success, frame = cap.read()
    if not success:
        break

    if frame_id % frames_to_skip == 0:
        minutes = int(frame_id / fps / 60)
        seconds = int(frame_id / fps % 60)
        print(f"\n⏱️ Frame [{frame_id}] at {minutes:02d}:{seconds:02d}")

        results = model(frame)
        person_count = 0
        for box in results[0].boxes:
            if int(box.cls[0]) == 0:
                person_count += 1
                x1, y1, x2, y2 = map(int, box.xyxy[0])
                cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)

        print(f"🧍 People detected: {person_count}")

        filename = os.path.join(saved_frames, f"frame_{minutes:02d}_{seconds:02d}.png")
        if cv2.imwrite(filename, frame):
            print(f"💾 Saved frame: {filename}")
        else:
            print("❌ Failed to save frame!")

    frame_id += 1

cap.release()
print("✅ Video processing complete.")


🎞️ FPS: 25.0

⏱️ Frame [0] at 00:00

0: 384x640 10 persons, 2 handbags, 1 suitcase, 552.0ms
Speed: 17.0ms preprocess, 552.0ms inference, 28.6ms postprocess per image at shape (1, 3, 384, 640)
🧍 People detected: 10
💾 Saved frame: saved_frames/frame_00_00.png

⏱️ Frame [25] at 00:01

0: 384x640 14 persons, 1 backpack, 1 handbag, 349.0ms
Speed: 4.4ms preprocess, 349.0ms inference, 1.3ms postprocess per image at shape (1, 3, 384, 640)
🧍 People detected: 14
💾 Saved frame: saved_frames/frame_00_01.png
✅ Video processing complete.


#step -3
using yolov8 and deepsort for calcuing the person standing behinf another person in the queue
this helps us to finbd whwther a person is standinmg in the line or not

In [ ]:
pip install deep_sort_realtime


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.4/8.4 MB 63.8 MB/s eta 0:00:00


now applying deepsort and checking ther model

for a 5 second video and stay duration is 2 second


In [ ]:
from ultralytics import YOLO
import cv2
import os
import numpy as np
from deep_sort_realtime.deepsort_tracker import DeepSort

# Load YOLOv8 model
model = YOLO("yolov8s.pt")

# Initialize DeepSORT tracker
tracker = DeepSort(max_age=30)

# Contrast enhancement using CLAHE
def enhance_contrast_brightness(frame):
    lab = cv2.cvtColor(frame, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    l_clahe = clahe.apply(l)
    merged = cv2.merge((l_clahe, a, b))
    return cv2.cvtColor(merged, cv2.COLOR_LAB2BGR)

# Load your test video
video_path = "video-1.mp4"
cap = cv2.VideoCapture(video_path)

if not cap.isOpened():
    print("❌ Could not load video.")
    exit()

fps = int(cap.get(cv2.CAP_PROP_FPS))

# Define ROI (area where queue is formed) - adjust this based on your video
roi_polygon = np.array([[150, 100], [450, 100], [450, 400], [150, 400]])

# Minimum stay = 2 seconds worth of frames
min_stay_frames = fps * 2  # For 30 FPS, this = 60 frames

# Track when each person ID enters the ROI
track_start_frame = {}
valid_ids = set()

# Save processed frames
save_path = "saved_frames"
os.makedirs(save_path, exist_ok=True)
frame_count = 0

while True:
    success, frame = cap.read()
    if not success:
        break

    enhanced = enhance_contrast_brightness(frame)
    results = model(enhanced)
    detections = []

    for box in results[0].boxes:
        if int(box.cls[0]) == 0:
            x1, y1, x2, y2 = map(int, box.xyxy[0])
            conf = float(box.conf[0])
            detections.append(([x1, y1, x2 - x1, y2 - y1], conf, 'person'))

    tracks = tracker.update_tracks(detections, frame=enhanced)

    for track in tracks:
        if not track.is_confirmed():
            continue

        track_id = track.track_id
        ltrb = track.to_ltrb()
        x1, y1, x2, y2 = map(int, ltrb)
        cx, cy = int((x1 + x2) / 2), int((y1 + y2) / 2)

        # Check if the person is in ROI
        if cv2.pointPolygonTest(roi_polygon, (cx, cy), False) >= 0:
            if track_id not in track_start_frame:
                track_start_frame[track_id] = frame_count
            elif frame_count - track_start_frame[track_id] >= min_stay_frames:
                valid_ids.add(track_id)

        # Draw bounding box and ID
        cv2.rectangle(enhanced, (x1, y1), (x2, y2), (0, 255, 0), 2)
        cv2.putText(enhanced, f"ID: {track_id}", (x1, y1 - 10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)

    # Draw ROI
    cv2.polylines(enhanced, [roi_polygon], isClosed=True, color=(0, 255, 255), thickness=2)

    # Save frame with clean name
    filename = os.path.join(save_path, f"frame-{frame_count + 1:02d}.png")
    cv2.imwrite(filename, enhanced)

    frame_count += 1

cap.release()

print("✅ Done processing video.")
print(f"🧍 People in queue > 2 seconds: {len(valid_ids)}")



0: 384x640 14 persons, 614.8ms
Speed: 3.2ms preprocess, 614.8ms inference, 2.5ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 14 persons, 405.8ms
Speed: 4.4ms preprocess, 405.8ms inference, 1.6ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 14 persons, 391.0ms
Speed: 3.2ms preprocess, 391.0ms inference, 1.5ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 14 persons, 390.9ms
Speed: 4.5ms preprocess, 390.9ms inference, 1.8ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 14 persons, 399.7ms
Speed: 4.0ms preprocess, 399.7ms inference, 1.5ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 14 persons, 405.9ms
Speed: 4.0ms preprocess, 405.9ms inference, 1.4ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 14 persons, 618.7ms
Speed: 3.2ms preprocess, 618.7ms inference, 1.8ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 14 persons, 627.8ms
Speed: 5.3ms preprocess, 627.8ms inference, 2.3ms postproc

| Feature                | Description                                                    |
| ---------------------- | -------------------------------------------------------------- |
| 🧍 YOLOv8 + DeepSORT   | Human detection + unique ID tracking                           |
| 🎯 Polygon ROI         | Only detect people inside the line area                        |
| ✅ Light Green Box      | For people **standing inside the queue**                       |
| ⚠️ Light Yellow Box    | For people **outside the queue** (chilling, chatting, walking) |
| 💡 CLAHE               | Low-light enhancement                                          |
| 💾 Frame save          | `frame-01.png`, `frame-02.png`, …                              |
| 🕓 2-second stay logic | For 5s video testing                                           |


In [ ]:
from ultralytics import YOLO
import cv2
import os
import numpy as np
from deep_sort_realtime.deepsort_tracker import DeepSort

# Load YOLOv8 model
model = YOLO("yolov8s.pt")

# Initialize DeepSORT tracker
tracker = DeepSort(max_age=30)

# Low-light enhancer
def enhance_contrast_brightness(frame):
    lab = cv2.cvtColor(frame, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    l_clahe = clahe.apply(l)
    merged = cv2.merge((l_clahe, a, b))
    return cv2.cvtColor(merged, cv2.COLOR_LAB2BGR)

# Load video
video_path = "video-1.mp4"
cap = cv2.VideoCapture(video_path)

if not cap.isOpened():
    print("❌ Could not load video.")
    exit()

fps = int(cap.get(cv2.CAP_PROP_FPS))

# Define ROI Polygon for queue area (adjust as per your camera angle)
roi_polygon = np.array([
    [130, 90],    # top-left
    [480, 90],    # top-right
    [520, 390],   # bottom-right
    [110, 390]    # bottom-left
])

# Stay logic
min_stay_frames = fps * 2  # 2 seconds = 60 frames

# For tracking
track_start_frame = {}
valid_ids = set()

# Save folder
save_path = "saved_frames"
os.makedirs(save_path, exist_ok=True)
frame_count = 0

while True:
    success, frame = cap.read()
    if not success:
        break

    enhanced = enhance_contrast_brightness(frame)
    results = model(enhanced)
    detections = []

    for box in results[0].boxes:
        if int(box.cls[0]) == 0:  # person
            x1, y1, x2, y2 = map(int, box.xyxy[0])
            conf = float(box.conf[0])
            detections.append(([x1, y1, x2 - x1, y2 - y1], conf, 'person'))

    tracks = tracker.update_tracks(detections, frame=enhanced)

    for track in tracks:
        if not track.is_confirmed():
            continue

        track_id = track.track_id
        ltrb = track.to_ltrb()
        x1, y1, x2, y2 = map(int, ltrb)
        cx, cy = int((x1 + x2) / 2), int((y1 + y2) / 2)

        in_queue = cv2.pointPolygonTest(roi_polygon, (cx, cy), False) >= 0

        # Logic: If inside ROI, track how long
        if in_queue:
            if track_id not in track_start_frame:
                track_start_frame[track_id] = frame_count
            elif frame_count - track_start_frame[track_id] >= min_stay_frames:
                valid_ids.add(track_id)

        # 🎨 Box color logic
        if track_id in valid_ids:
            color = (144, 238, 144)  # light green (in line > 2s)
        else:
            color = (0, 255, 255)  # light yellow (outside line or < 2s)

        cv2.rectangle(enhanced, (x1, y1), (x2, y2), color, 2)
        cv2.putText(enhanced, f"ID: {track_id}", (x1, y1 - 10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)

    # Draw the ROI queue polygon
    cv2.polylines(enhanced, [roi_polygon], isClosed=True, color=(0, 255, 255), thickness=2)

    # Save the frame
    filename = os.path.join(save_path, f"frame-{frame_count + 1:02d}.png")
    cv2.imwrite(filename, enhanced)

    frame_count += 1

cap.release()

print("✅ Video processing done.")
print(f"🧍 People confirmed in line (>2s): {len(valid_ids)}")



0: 384x640 14 persons, 403.0ms
Speed: 2.8ms preprocess, 403.0ms inference, 29.4ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 14 persons, 333.7ms
Speed: 3.0ms preprocess, 333.7ms inference, 1.3ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 14 persons, 554.2ms
Speed: 3.1ms preprocess, 554.2ms inference, 1.7ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 14 persons, 545.6ms
Speed: 3.0ms preprocess, 545.6ms inference, 2.1ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 14 persons, 333.0ms
Speed: 2.8ms preprocess, 333.0ms inference, 1.3ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 14 persons, 332.4ms
Speed: 2.9ms preprocess, 332.4ms inference, 1.2ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 14 persons, 343.6ms
Speed: 2.9ms preprocess, 343.6ms inference, 1.8ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 14 persons, 342.6ms
Speed: 2.9ms preprocess, 342.6ms inference, 1.3ms postpro